In [11]:
import openpyxl
file_path=r"C:\Users\Hi\OneDrive\Desktop\CAP776\12628918.xlsx"
wb= openpyxl.load_workbook(file_path)
ws = wb.active

from openpyxl import load_workbook
from datetime import date, datetime
from collections import Counter
from statistics import mean, correlation
from math import isfinite

FILE_PATH = r"C:\Users\Hi\OneDrive\Desktop\CAP776\12628918.xlsx"
SHEET_NAME = "Daily Log"
START_DATE = date(2026, 8, 13)
END_DATE = date(2026, 9, 25)
MINUTES_PER_DAY = 1440

# These mappings follow the scales described in the project report.
FEELING_SCORES = {"Excellent": 5, "Good": 4, "Neutral": 3, "Low": 2, "Stressed": 1}
SATISFACTION_SCORES = {"Very Satisfied": 5, "Satisfied": 4, "Neutral": 3, "Unsatisfied": 2, "Very Unsatisfied": 1}
ENERGY_SCORES = {"High": 3, "Medium": 2, "Low": 1}

EXPECTED_DAYS = (END_DATE - START_DATE).days + 1
print(f"Expected recording days: {EXPECTED_DAYS} ({START_DATE} to {END_DATE})")

def as_date(value):
    """Convert an Excel date/datetime to date; return None for other values."""
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    return None

def number_or_error(value, label):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError(f"{label} must be a number")
    if not isfinite(value) or value < 0:
        raise ValueError(f"{label} must be a finite, non-negative number")
    return float(value)

def load_activity_rows(file_path):
    workbook = load_workbook(file_path, data_only=True, read_only=True)
    if SHEET_NAME not in workbook.sheetnames:
        raise KeyError(f"Sheet '{SHEET_NAME}' not found. Available sheets: {workbook.sheetnames}")
    sheet = workbook[SHEET_NAME]
    headers = [cell.value for cell in sheet[5]]
    expected_headers = ["Date", "Sleep (min)", "Fitness (min)", "Study (min)",
                        "Coding (min)", "Class (min)", "Classes Attended",
                        "Other Activities (min)", "Total Tracked (min)",
                        "Free/Unaccounted (min)", "Day's Feeling",
                        "Satisfaction Level", "Energy Level", "Notes"]
    if headers[:len(expected_headers)] != expected_headers:
        raise ValueError("Unexpected column headings in row 5. Check that this is the supplied Daily Log template.")

    candidates = []
    invalid_records = []
    for excel_row, row in enumerate(sheet.iter_rows(min_row=6, values_only=True), start=6):
        if not row or all(value is None for value in row):
            continue
        logged_date = as_date(row[0])
        # Ignore the template sample and any entries outside the requested project period.
        if logged_date is None or not (START_DATE <= logged_date <= END_DATE):
            continue
        candidates.append((excel_row, logged_date, row))

    date_counts = Counter(item[1] for item in candidates)
    valid_rows = []
    for excel_row, logged_date, row in candidates:
        errors = []
        if date_counts[logged_date] > 1:
            errors.append("duplicate date; all rows for this date are excluded")
        try:
            sleep = number_or_error(row[1], "Sleep")
            fitness = number_or_error(row[2], "Fitness")
            study = number_or_error(row[3], "Study")
            coding = number_or_error(row[4], "Coding")
            class_time = number_or_error(row[5], "Class")
            other = number_or_error(row[7], "Other Activities")
        except ValueError as error:
            errors.append(str(error))
            sleep = fitness = study = coding = class_time = other = None
        feeling = str(row[10]).strip() if row[10] is not None else ""
        satisfaction = str(row[11]).strip() if row[11] is not None else ""
        energy = str(row[12]).strip() if row[12] is not None else ""
        if feeling not in FEELING_SCORES:
            errors.append(f"unrecognized/missing feeling: {feeling!r}")
        if satisfaction not in SATISFACTION_SCORES:
            errors.append(f"unrecognized/missing satisfaction: {satisfaction!r}")
        if energy not in ENERGY_SCORES:
            errors.append(f"unrecognized/missing energy: {energy!r}")

        if not errors:
            total_tracked = sleep + fitness + study + coding + class_time + other
            if total_tracked > MINUTES_PER_DAY:
                errors.append(f"tracked time {total_tracked:g} exceeds 1,440 minutes")
        if errors:
            invalid_records.append((excel_row, logged_date, errors))
            continue

        valid_rows.append({
            "date": logged_date, "sleep": sleep, "fitness": fitness,
            "study": study, "coding": coding, "class": class_time,
            "other": other, "total_tracked": total_tracked,
            "free_time": MINUTES_PER_DAY - total_tracked,
            "feeling": feeling, "satisfaction": satisfaction, "energy": energy,
            "feeling_score": FEELING_SCORES[feeling],
            "satisfaction_score": SATISFACTION_SCORES[satisfaction],
            "energy_score": ENERGY_SCORES[energy],
        })
    workbook.close()
    return valid_rows, invalid_records

try:
    daily_records, invalid_records = load_activity_rows(FILE_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find workbook: {FILE_PATH}. Update FILE_PATH and run this cell again.")

if not daily_records:
    raise ValueError("No valid activity rows found in the selected date range. Check the workbook and validation messages.")

recorded_dates = {record["date"] for record in daily_records}
missing_dates = [START_DATE.fromordinal(day) for day in range(START_DATE.toordinal(), END_DATE.toordinal() + 1)
                 if START_DATE.fromordinal(day) not in recorded_dates]
print(f"Valid recorded days: {len(daily_records)}")
print(f"Missing days: {len(missing_dates)}")
print(f"Invalid/excluded in-period rows: {len(invalid_records)}")
if invalid_records:
    print("\nInvalid rows:")
    for row_number, logged_date, errors in invalid_records:
        print(f"  Excel row {row_number} ({logged_date}): {'; '.join(errors)}")
if missing_dates:
    print("\nDates with no valid record:", ", ".join(day.isoformat() for day in missing_dates))

def average_field(records, field):
    return mean(record[field] for record in records)

def calculate_indices(records, expected_days):
    if not records:
        raise ValueError("Cannot calculate averages without valid recorded days.")
    tpi = average_field(records, "coding")
    aai = mean(record["study"] + record["class"] for record in records)
    phai = average_field(records, "fitness")
    sri = average_field(records, "sleep")
    abi = average_field(records, "free_time")
    tui = average_field(records, "total_tracked")
    ei = mean((record["feeling_score"] + record["satisfaction_score"] + record["energy_score"]) / 3
              for record in records)
    dci = len(records) / expected_days * 100
    pai = (0.15 * tpi + 0.20 * aai + 0.15 * phai + 0.20 * sri
           + 0.15 * tui + 0.10 * ei + 0.05 * dci)
    return {"TPI": tpi, "AAI": aai, "PhAI": phai, "SRI": sri,
            "ABI": abi, "TUI": tui, "EI": ei, "DCI": dci, "PAI": pai}

indices = calculate_indices(daily_records, EXPECTED_DAYS)
print("Average daily activity (minutes/day)")
for label, key in [("Sleep", "sleep"), ("Fitness", "fitness"),
                   ("Study", "study"), ("Coding", "coding"),
                   ("Class", "class"), ("Other Activities", "other"),
                   ("Free/Unaccounted", "free_time")]:
    print(f"  {label}: {average_field(daily_records, key):.2f}")
print(f"\nInvalid/excluded row count: {len(invalid_records)}")
print("\nIndices")
for name in ("TPI", "AAI", "PhAI", "SRI", "ABI", "TUI"):
    print(f"  {name}: {indices[name]:.2f} min/day")
print(f"  EI: {indices['EI']:.2f} / 5 (per report formula)")
print(f"  DCI: {indices['DCI']:.2f}%")
print(f"  PAI: {indices['PAI']:.2f} (report's weighted formula)")

def safe_correlation(records, x_key, y_key):
    x_values = [record[x_key] for record in records]
    y_values = [record[y_key] for record in records]
    if len(records) < 2:
        return None
    if len(set(x_values)) < 2 or len(set(y_values)) < 2:
        return None
    return correlation(x_values, y_values)

relationships = [
    ("Sleep ↔ Energy", "sleep", "energy_score"),
    ("Study ↔ Satisfaction", "study", "satisfaction_score"),
    ("Coding ↔ Energy", "coding", "energy_score"),
]
print("Pearson correlations (valid recorded days only)")
for label, x_key, y_key in relationships:
    value = safe_correlation(daily_records, x_key, y_key)
    if value is None:
        print(f"  {label}: not available (need at least two days and variation in both values)")
    else:
        direction = "positive" if value > 0 else "negative" if value < 0 else "no linear association"
        print(f"  {label}: r = {value:.3f} ({direction} association)")

print("\nUse these results as observations from this log; correlation alone does not show causation.")

Expected recording days: 44 (2026-08-13 to 2026-09-25)
Valid recorded days: 41
Missing days: 3
Invalid/excluded in-period rows: 0

Dates with no valid record: 2026-09-23, 2026-09-24, 2026-09-25
Average daily activity (minutes/day)
  Sleep: 372.68
  Fitness: 0.00
  Study: 159.76
  Coding: 123.90
  Class: 179.27
  Other Activities: 64.02
  Free/Unaccounted: 540.37

Invalid/excluded row count: 0

Indices
  TPI: 123.90 min/day
  AAI: 339.02 min/day
  PhAI: 0.00 min/day
  SRI: 372.68 min/day
  ABI: 540.37 min/day
  TUI: 899.63 min/day
  EI: 3.45 / 5 (per report formula)
  DCI: 93.18%
  PAI: 300.88 (report's weighted formula)
Pearson correlations (valid recorded days only)
  Sleep ↔ Energy: r = -0.320 (negative association)
  Study ↔ Satisfaction: r = -0.137 (negative association)
  Coding ↔ Energy: r = 0.196 (positive association)

Use these results as observations from this log; correlation alone does not show causation.
